# LangGraph streaming modes with DeepSeek

这个 notebook 独立演示 LangGraph 的 7 种 streaming mode：`updates`、`values`、`messages`、`custom`、`tasks`、`checkpoints`、`debug`。

示例风格参考 `examples/workflows_agents_deepseek.ipynb`，但这里的目标不是讲 workflow pattern，而是专门把 streaming 当成 runtime 的 7 扇观察窗来理解。

## 七种观察窗

| 观察窗 | stream mode | 你会看到什么 | 它更像在回答什么问题 |
|---|---|---|---|
| 状态增量 | `updates` | 每个节点或 task 刚刚写入了什么 | 谁改了什么？ |
| 状态快照 | `values` | 每一步之后的完整 state | 全局状态现在长什么样？ |
| 模型输出 | `messages` | LLM 的 token / message chunk 以及对应 metadata | 模型此刻在输出什么？是哪次调用、哪个 node 发出来的？ |
| 业务事件 | `custom` | 节点通过 `StreamWriter` 主动发出的任意数据 | 这一步想对外汇报什么进度或信号？ |
| 运行时事件 | `tasks` | task 的开始、结束、结果和错误 | 现在跑到哪了？哪里卡住或失败了？ |
| 运行时事件 | `checkpoints` | checkpoint 创建时的事件和对应数据 | 什么时候形成了一个可恢复的保存点？ |
| 深度调试 | `debug` | 比 `tasks` / `checkpoints` 更完整的调试视图 | 如果要还原现场，runtime 当时到底看到了什么？ |

为了让 7 种 mode 的输出更统一，下面的示例统一使用 `version="v2"`，这样每条事件都带 `type`、`ns` 和 `data`。

## Setup

建议使用 `langgraph-deepseek` 内核。缺依赖时，可取消下一格的注释后执行安装。

这个 notebook 不会写死 API key。运行模型相关 cell 之前，请先在环境变量里设置 `DEEPSEEK_API_KEY`。

In [ ]:
# %pip install -U langgraph langchain langchain-openai pydantic

In [ ]:
import operator
import os
import uuid
from pprint import pprint
from typing import Annotated

from IPython.display import Image, display
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import StreamWriter
from typing_extensions import TypedDict

In [ ]:
os.environ.setdefault("DEEPSEEK_BASE_URL", "https://api.deepseek.com")
os.environ.setdefault("DEEPSEEK_MODEL", "deepseek-v4-flash")

if "DEEPSEEK_API_KEY" not in os.environ:
    raise ValueError(
        "DEEPSEEK_API_KEY is not set. Export it before running this notebook, "
        "or define it in a separate setup cell."
    )

streaming_llm = ChatOpenAI(
    model=os.environ["DEEPSEEK_MODEL"],
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url=os.environ["DEEPSEEK_BASE_URL"],
    temperature=0,
    streaming=True,
)

## Build one demo graph

这张图故意很小，但同时覆盖三类信号：

- 节点 state 写入：用于看 `updates` 和 `values`
- DeepSeek 模型调用：用于看 `messages`
- 节点主动发出的进度事件：用于看 `custom`

另外，图会直接挂一个 `InMemorySaver()`，这样 `tasks`、`checkpoints` 和 `debug` 也能一起观测。

In [ ]:
class StreamingDemoState(TypedDict):
    topic: str
    outline: str
    draft: str
    final_answer: str
    messages: Annotated[list[BaseMessage], add_messages]
    audit: Annotated[list[str], operator.add]


def plan_step(state: StreamingDemoState, *, writer: StreamWriter):
    writer({"event": "plan_started", "topic": state["topic"]})
    outline = "\n".join(
        [
            "1. 解释 streaming 为什么不只是 token 输出",
            "2. 展示 runtime 如何暴露 state 与 task 变化",
            "3. 用一句业务视角的话收尾",
        ]
    )
    writer({"event": "plan_finished", "outline_lines": 3})
    return {"outline": outline, "audit": ["plan"]}


def draft_step(state: StreamingDemoState, *, writer: StreamWriter):
    writer(
        {
            "event": "draft_started",
            "outline_preview": state["outline"].splitlines(),
        }
    )
    response = streaming_llm.invoke(
        [
            HumanMessage(
                content=(
                    f"请用中文写一个紧凑、具体的自然段，主题是：{state['topic']}。"
                    "不要下定义式空话，要强调运行时可观测性。"
                    f"请参考这个提纲：\n{state['outline']}"
                )
            )
        ]
    )
    writer({"event": "draft_finished", "chars": len(response.content)})
    return {
        "draft": response.content,
        "messages": [response],
        "audit": ["draft"],
    }


def finalize_step(state: StreamingDemoState, *, writer: StreamWriter):
    writer({"event": "finalize_started"})
    final_answer = "\n".join(
        [
            "观察总结：",
            "- outline 已经进入 state，适合看 updates / values",
            "- draft 已由 DeepSeek 生成，适合看 messages",
            f"- audit 执行路径：{state['audit'] + ['finalize']}"
        ]
    )
    writer({"event": "finalize_finished", "final_chars": len(final_answer)})
    return {"final_answer": final_answer, "audit": ["finalize"]}


streaming_builder = StateGraph(StreamingDemoState)
streaming_builder.add_node("plan_step", plan_step)
streaming_builder.add_node("draft_step", draft_step)
streaming_builder.add_node("finalize_step", finalize_step)
streaming_builder.add_edge(START, "plan_step")
streaming_builder.add_edge("plan_step", "draft_step")
streaming_builder.add_edge("draft_step", "finalize_step")
streaming_builder.add_edge("finalize_step", END)

streaming_graph = streaming_builder.compile(checkpointer=InMemorySaver())

STREAMING_INPUT = {
    "topic": "为什么 LangGraph 的 streaming 更像 runtime 观察窗，而不仅仅是 token 输出",
    "outline": "",
    "draft": "",
    "final_answer": "",
    "messages": [],
    "audit": [],
}

display(Image(streaming_graph.get_graph().draw_mermaid_png()))

In [ ]:
def make_stream_config(label: str):
    return {
        "configurable": {
            "thread_id": f"stream-demo-{label}-{uuid.uuid4().hex[:8]}"
        }
    }


def collect_mode(mode: str):
    return list(
        streaming_graph.stream(
            STREAMING_INPUT,
            config=make_stream_config(mode),
            stream_mode=mode,
            version="v2"
        )
    )


def extract_message_text(message_chunk):
    text_value = getattr(message_chunk, "text", "")
    if callable(text_value):
        text_value = text_value()
    if text_value:
        return text_value

    content = getattr(message_chunk, "content", "")
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        text_parts = []
        for item in content:
            if isinstance(item, str):
                text_parts.append(item)
            elif isinstance(item, dict) and "text" in item:
                text_parts.append(str(item["text"]))
        return "".join(text_parts)

    return str(content) if content else ""


def show_mode(mode: str, *, limit: int = 5):
    chunks = collect_mode(mode)
    print(f"stream_mode={mode!r}, total_chunks={len(chunks)}")
    print("=" * 80)

    if mode == "messages":
        message_preview = []
        for chunk in chunks:
            message_chunk, metadata = chunk["data"]
            chunk_text = extract_message_text(message_chunk)
            if not chunk_text:
                continue
            message_preview.append(
                {
                    "type": chunk["type"],
                    "ns": chunk["ns"],
                    "chunk_type": type(message_chunk).__name__,
                    "chunk_text": chunk_text,
                    "node": metadata.get("langgraph_node"),
                    "step": metadata.get("langgraph_step")
                }
            )

        for item in message_preview[:limit]:
            pprint(item)
            print("-" * 80)

        print(f"non_empty_text_chunks={len(message_preview)} / total_events={len(chunks)}")
        if len(message_preview) > limit:
            print(f"... {len(message_preview) - limit} more text chunk(s)")
        return chunks

    for chunk in chunks[:limit]:
        if mode == "tasks":
            payload = chunk["data"]
            pprint(
                {
                    "type": chunk["type"],
                    "name": payload.get("name"),
                    "id": payload.get("id"),
                    "triggers": payload.get("triggers"),
                    "has_result": "result" in payload,
                    "error": payload.get("error")
                }
            )
        elif mode == "checkpoints":
            payload = chunk["data"]
            pprint(
                {
                    "type": chunk["type"],
                    "next": payload["next"],
                    "values_keys": sorted(payload["values"].keys()),
                    "task_count": len(payload["tasks"])
                }
            )
        elif mode == "debug":
            envelope = chunk["data"]
            pprint(
                {
                    "type": chunk["type"],
                    "event_type": envelope["type"],
                    "step": envelope["step"],
                    "payload_keys": sorted(envelope["payload"].keys())
                }
            )
        else:
            pprint(chunk)
        print("-" * 80)

    if len(chunks) > limit:
        print(f"... {len(chunks) - limit} more chunk(s)")
    return chunks

## One graph, seven observation windows

下面每个 cell 都会重新跑同一张图，只是把 `stream_mode` 换成不同值。这样你可以直接横向对比它们分别在暴露哪一层 runtime 信息。

In [ ]:
updates_events = show_mode("updates")
updates_events

In [ ]:
values_events = show_mode("values")
values_events

In [ ]:
messages_events = show_mode("messages", limit=8)
messages_events

In [ ]:
custom_events = show_mode("custom")
custom_events

In [ ]:
tasks_events = show_mode("tasks", limit=8)
tasks_events

In [ ]:
checkpoint_events = show_mode("checkpoints", limit=8)
checkpoint_events

In [ ]:
debug_events = show_mode("debug", limit=10)
debug_events

## Optional: subscribe to several windows at once

如果你想把多个观察窗一起接到同一个消费者上，可以把 `stream_mode` 传成列表。最外层仍然是一条统一事件流，只是 `type` 会告诉你当前事件来自哪扇观察窗。

In [ ]:
mixed_events = list(
    streaming_graph.stream(
        STREAMING_INPUT,
        config=make_stream_config("mixed"),
        stream_mode=["updates", "messages", "custom"],
        version="v2"
    )
)

for event in mixed_events[:12]:
    print(event["type"])
    pprint(event)
    print("-" * 80)